# DACS-SCV v3: transparency-safe resumable formal SciVer runner

This notebook installs the background package from a writable directory, checks the public release, executes the frozen five-seed formal protocol, and packages every audit artifact. The default is the **formal run on Kaggle** and a **synthetic smoke run elsewhere**. Version 3 correctly composites transparent scientific figures onto white before perceptual auditing and CLIP encoding, while preserving downloads, features, and checkpoints in persistent `/kaggle/working` locations. The formal archive still requires independent audit before its values enter the paper.

## 1. Parameters

The local development pilots and protocol audit are complete. On Kaggle, leave `PROFILE='formal'` unchanged. Before running, add a private Kaggle secret named `HF_TOKEN` containing a free Hugging Face **read-only** token; the notebook reads it without printing it. Formal mode uses all eligible public SciVer development records, five predetermined seeds, frozen BGE/CLIP features, and 10,000 source-paper-grouped bootstrap samples on the official test. Do not rerun with altered settings after viewing the test output.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

INPUT_ROOT = Path(os.environ.get('TRACE_SCV_INPUT_ROOT', '/kaggle/input'))
WORKING_ROOT = Path(os.environ.get('TRACE_SCV_WORKING_ROOT', '/kaggle/working'))
IS_KAGGLE = os.environ.get('TRACE_SCV_FORCE_KAGGLE') == '1' or Path('/kaggle/input').exists()
PROFILE = os.environ.get('TRACE_SCV_PROFILE', 'formal' if IS_KAGGLE else 'smoke')  # smoke | formal
DOWNLOAD_PUBLIC_DATA = PROFILE != 'smoke'
DEVICE = 'auto'
EXPECTED_VERSION = '0.3.2'
DATA_ROOT = Path(os.environ.get('TRACE_SCV_DATA_ROOT', WORKING_ROOT / 'dacs-scv-data-v2'))
OUTPUT_ROOT = Path(os.environ.get('TRACE_SCV_OUTPUT_ROOT', WORKING_ROOT / 'dacs-scv-outputs-v2'))
os.environ['TRACE_SCV_DATA_ROOT'] = str(DATA_ROOT)
os.environ['TRACE_SCV_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '600')
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '120')
os.environ.setdefault('TRACE_SCV_HF_RETRIES', '5')
os.environ.setdefault('TRACE_SCV_HF_WORKERS', '2')

if IS_KAGGLE and not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        if secret_token:
            os.environ['HF_TOKEN'] = secret_token
    except Exception:
        pass
run_configuration = {
    'profile': PROFILE,
    'download': DOWNLOAD_PUBLIC_DATA,
    'kaggle': IS_KAGGLE,
    'hf_authenticated': bool(os.environ.get('HF_TOKEN')),
    'resumable_data_root': str(DATA_ROOT),
    'resumable_output_root': str(OUTPUT_ROOT),
}
print(run_configuration)
if IS_KAGGLE and PROFILE == 'formal' and not os.environ.get('HF_TOKEN'):
    raise RuntimeError(
        'Missing Kaggle secret HF_TOKEN. Add a free Hugging Face read-only token '
        'under Add-ons > Secrets, enable it for this notebook, restart the session, '
        'and choose Run All. Do not paste the token into a cell.'
    )

## 2. Locate and install the code

Kaggle inputs are read-only. This setup validates the uploaded source, copies or extracts it to `/kaggle/working`, installs it non-editably, and then pins Python to that exact source tree. The pin and module-cache reset prevent an older `trace_scv` package from shadowing the uploaded code.

In [ ]:
REQUIRED_SOURCE_FILES = (
    Path('experiments/pyproject.toml'),
    Path('experiments/src/trace_scv/data/__init__.py'),
    Path('experiments/src/trace_scv/data/preflight.py'),
    Path('experiments/src/trace_scv/cardinality.py'),
    Path('experiments/src/trace_scv/models/cardinality.py'),
    Path('experiments/src/trace_scv/pipeline.py'),
)

def valid_repository(candidate: Path) -> bool:
    if not all((candidate / relative).is_file() for relative in REQUIRED_SOURCE_FILES):
        return False
    version_file = candidate / 'experiments/src/trace_scv/__init__.py'
    return f'__version__ = "{EXPECTED_VERSION}"' in version_file.read_text(encoding='utf-8')

def repository_roots_below(search_root: Path) -> list[Path]:
    if not search_root.exists():
        return []
    roots = []
    for marker in search_root.rglob('pyproject.toml'):
        if marker.parent.name != 'experiments':
            continue
        candidate = marker.parent.parent
        if valid_repository(candidate):
            roots.append(candidate.resolve())
    return sorted(set(roots), key=lambda path: str(path), reverse=True)

def safely_extract(archive_path: Path, destination: Path) -> None:
    with zipfile.ZipFile(archive_path) as handle:
        destination_root = destination.resolve()
        for member in handle.infolist():
            target = (destination / member.filename).resolve()
            if target != destination_root and destination_root not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member in {archive_path}: {member.filename}')
        handle.extractall(destination)

def find_repository() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if valid_repository(candidate):
            return candidate.resolve()
    for candidate in repository_roots_below(INPUT_ROOT):
        print('Selected source directory:', candidate)
        return candidate
    if INPUT_ROOT.exists():
        archives = sorted(
            (path for path in INPUT_ROOT.rglob('*') if path.is_file() and path.suffix.lower() == '.zip' and 'public-data' not in path.name.lower()),
            key=lambda path: str(path),
            reverse=True,
        )
        for archive_path in archives:
            if not zipfile.is_zipfile(archive_path):
                continue
            extracted = WORKING_ROOT / 'dacs-scv-source-v3'
            if extracted.exists():
                shutil.rmtree(extracted)
            extracted.mkdir(parents=True, exist_ok=True)
            safely_extract(archive_path, extracted)
            extracted_roots = repository_roots_below(extracted)
            if extracted_roots:
                print('Selected archive:', archive_path)
                return extracted_roots[0]
    visible = []
    if INPUT_ROOT.exists():
        visible = sorted(
            str(path.relative_to(INPUT_ROOT))
            for path in INPUT_ROOT.rglob('*')
            if path.is_file()
        )[:80]
    raise FileNotFoundError(
        f'No complete DACS-SCV v3 ({EXPECTED_VERSION}) upload was found under '
        f'{INPUT_ROOT}. Visible files (first 80): {visible}'
    )

source_root = find_repository()

def repository_copy_ignore(directory: str, names: list[str]) -> set[str]:
    # Ignore generated artifacts everywhere, but ignore experiments/data only
    # at that exact level. Never ignore trace_scv/data, which is source code.
    ignored = {name for name in names if name in {'.git', '.venv', '.pytest_cache', '__pycache__', 'outputs', 'build'} or name.endswith('.egg-info')}
    if Path(directory).resolve() == (source_root / 'experiments').resolve():
        ignored.add('data')
    return ignored

try:
    source_root.relative_to(INPUT_ROOT.resolve())
    source_is_read_only_input = True
except ValueError:
    source_is_read_only_input = False
if source_is_read_only_input:
    repo_root = WORKING_ROOT / 'dacs-scv'
    if repo_root.exists():
        shutil.rmtree(repo_root)
    shutil.copytree(source_root, repo_root, ignore=repository_copy_ignore)
else:
    repo_root = source_root
package_root = repo_root / 'experiments'
missing = [str(relative) for relative in REQUIRED_SOURCE_FILES if not (repo_root / relative).is_file()]
assert not missing, f'Incomplete DACS-SCV upload; missing: {missing}'
print('Repository:', repo_root)

In [ ]:
import importlib

extras = 'formal,analysis' if PROFILE in {'pilot', 'formal'} else 'ml,data,analysis'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
    '--upgrade-strategy', 'only-if-needed', f'{package_root}[{extras}]'
], check=True)

# A restarted Kaggle kernel may retain an older TRACE-SCV module. Import the
# uploaded source explicitly and clear every cached submodule before preflight.
source_dir = (package_root / 'src').resolve()
sys.path = [entry for entry in sys.path if Path(entry or '.').resolve() != source_dir]
sys.path.insert(0, str(source_dir))
for module_name in list(sys.modules):
    if module_name == 'trace_scv' or module_name.startswith('trace_scv.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import trace_scv
loaded_package = Path(trace_scv.__file__).resolve().parent
expected_package = source_dir / 'trace_scv'
assert loaded_package == expected_package, (loaded_package, expected_package)
assert (loaded_package / 'data' / 'preflight.py').is_file()
assert trace_scv.__version__ == EXPECTED_VERSION, (trace_scv.__version__, EXPECTED_VERSION)
print('DACS-SCV import verified:', loaded_package)
print('Any dopamine-rl/gym resolver warning above is from Kaggle preinstalled packages; DACS-SCV does not import either package.')

## 3. Verify public releases

In [ ]:
from trace_scv.data.preflight import run_preflight
preflight = run_preflight()
assert preflight['status'] == 'pass', preflight
[(row['name'], row['accessible'], row.get('revision')) for row in preflight['datasets']]

## 4. Run the experiment

The runner performs normalization, dual-channel semantic/visual feature construction, base-verifier training, frozen-ranking fixed-budget comparisons, decoupled cardinality training, official-test inference, grouped bootstrap comparisons, and output export.

In [ ]:
from trace_scv.pipeline import run_pipeline
audit_dir = OUTPUT_ROOT / PROFILE / 'audit'
audit_dir.mkdir(parents=True, exist_ok=True)
(audit_dir / 'preflight.json').write_text(json.dumps(preflight, indent=2), encoding='utf-8')
result = run_pipeline(repo_root, profile=PROFILE, download=DOWNLOAD_PUBLIC_DATA, device=DEVICE)
{'summary': result['summary'], 'pilot_readiness_gate': result.get('pilot_readiness_gate')}

## 5. Review bounded results

In [ ]:
import pandas as pd
summary = pd.DataFrame(result['summary']).T.reset_index(names='variant')
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt
plot = summary.set_index('variant')[['verdict_macro_f1_mean', 'citation_macro_f1_mean']].plot(kind='bar', figsize=(8, 4), color=['#2f6b9a', '#d17c2f'])
plot.set_title(f'DACS-SCV {PROFILE} profile: verdict and citation metrics')
plot.set_ylabel('Score (0 to 1)')
plot.set_ylim(0, 1)
plot.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Export outputs

The ZIP contains preflight and dataset manifests, cross-split duplicate records, input/code/configuration hashes, base and cardinality checkpoints, complete rankings, predictions, per-claim comparison records, grouped bootstrap comparisons, an artifact inventory, and the aggregate summary. Return the formal ZIP for independent audit before copying any number into the manuscript.

In [ ]:
output_dir = OUTPUT_ROOT / PROFILE
archive_base = WORKING_ROOT / f'dacs-scv-{PROFILE}-outputs' if IS_KAGGLE else repo_root / f'dacs-scv-{PROFILE}-outputs'
archive = Path(shutil.make_archive(str(archive_base), 'zip', output_dir))
print('Output archive:', archive)
print('Scientific validation flag:', result['scientific_results_validated'])

## 7. Interpretation

- `smoke`: validates software behavior only.
- `smoke`: synthetic software validation only; it is never scientific evidence.
- `formal`: executes the single frozen public-data matrix. Audit its manifests, predictions, ranking/verdict identity checks, and uncertainty outputs before transferring any number into the paper.